# Employee Attrition Analysis

## Problem Statement

Employee attrition can lead to increased hiring costs, loss of productivity, and disruption to business operations. The objective of this project is to analyze employee data and identify factors associated with employee attrition.

The analysis focuses on age groups, salary, overtime, and job satisfaction to understand which factors may contribute to employees leaving the company.


Installing PYmsql to connect AWS RDS with Collab

In [1]:
!pip install pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.6 MB/s eta 0:00:00


Imprting necesary libraries

In [2]:
import pandas as pd
import pymysql


Connecting Database to Collab

In [4]:
conn=pymysql.connect(
    host="",
    user="",
    password="",
    database="",
)

In [5]:
pd.read_sql("SHOW TABLES;", conn)

/tmp/ipykernel_4812/3845684567.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SHOW TABLES;", conn)


,Tables_in_employee_attrition
0,employee


In [6]:
df=pd.read_sql("select Attrition,age,BusinessTravel,Department,gender,JobSatisfaction,monthlyincome,overtime from employee",conn)

/tmp/ipykernel_4812/3902146766.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql("select Attrition,age,BusinessTravel,Department,gender,JobSatisfaction,monthlyincome,overtime from employee",conn)


In [7]:
df

,Attrition,age,BusinessTravel,Department,gender,JobSatisfaction,monthlyincome,overtime
0,Yes,41,Travel_Rarely,Sales,Female,4,5993,Yes
1,No,49,Travel_Frequently,Research & Development,Male,2,5130,No
2,Yes,37,Travel_Rarely,Research & Development,Male,3,2090,Yes
3,No,33,Travel_Frequently,Research & Development,Female,3,2909,Yes
4,No,27,Travel_Rarely,Research & Development,Male,2,3468,No
...,...,...,...,...,...,...,...,...
1465,No,36,Travel_Frequently,Research & Development,Male,4,2571,No
1466,No,39,Travel_Rarely,Research & Development,Male,1,9991,No
1467,No,27,Travel_Rarely,Research & Development,Male,2,6142,Yes
1468,No,49,Travel_Frequently,Sales,Male,2,5390,No


## Overall Attrition Analysis

### Why this analysis?

Before investigating the causes of employee attrition, it is important to determine the overall attrition rate and understand the scale of the problem within the company.

In [8]:
df[["Attrition"]].value_counts()

,count
Attrition,
No,1233
Yes,237


Creating Age_Bands

In [9]:
youngest_age=df["age"].min()
oldest_age=df["age"].max()

In [10]:
print("youngest_age =", youngest_age)
print("oldest_age =", oldest_age)

youngest_age = 18
oldest_age = 60


In [11]:
df["Age_Band"] = pd.cut(
    df["age"],
    bins=[17, 30, 50, 60],
    labels=["Rookie(below 30)", "Experienced(30-50)", "Senior(50-60)"]
)

## Age Analysis

### Why this analysis?

After identifying the overall attrition rate, the next step is to determine whether attrition is concentrated within a particular age group. This helps identify high-risk employee segments that may require further investigation.

In [12]:
pd.crosstab(df["Age_Band"], df["Attrition"])

Attrition,No,Yes
Age_Band,,
Rookie(below 30),286,100
Experienced(30-50),822,119
Senior(50-60),125,18


In [13]:
pd.crosstab(
    df["Age_Band"],
    df["Attrition"],
    normalize="index"
) * 100

Attrition,No,Yes
Age_Band,,
Rookie(below 30),74.093264,25.906736
Experienced(30-50),87.353879,12.646121
Senior(50-60),87.412587,12.587413


### Finding

Rookie employees have an attrition rate of 25.9%, compared to 12.7% for experienced employees and 12.6% for senior employees.

Although experienced employees account for more departures in absolute numbers (119), rookie employees are significantly more likely to leave relative to their group size.

## Salary Analysis

### Why this analysis?

The age analysis showed that rookie employees have the highest attrition rate. One possible explanation is compensation. This analysis investigates whether salary differences may be associated with rookie attrition.

###Salary by Age Group

In [14]:
df.groupby("Age_Band")["monthlyincome"].mean()

/tmp/ipykernel_4812/743201045.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Age_Band")["monthlyincome"].mean()


,monthlyincome
Age_Band,
Rookie(below 30),3866.829016
Experienced(30-50),6965.467588
Senior(50-60),10574.881119


Rookie employees earn substantially less than experienced and senior employees.

### Salary Comparison Within Rookie Employees

To further investigate the relationship between salary and attrition, rookie employees were separated into two groups: those who stayed and those who left the company.

The average monthly income of each group was then compared.

In [15]:
rookies = df[df["Age_Band"] == "Rookie(below 30)"]

In [16]:
rookies.groupby("Attrition")["monthlyincome"].mean()

,monthlyincome
Attrition,
No,4142.916084
Yes,3077.220000


### Finding

Rookie employees who left earned an average monthly income of 3077, compared to 4143 for rookie employees who stayed.

This difference suggests that lower compensation may be associated with higher attrition among rookie employees.

## Overtime Analysis

### Why this analysis?

The age analysis revealed that rookie employees experience significantly higher attrition than other groups. One possible explanation is that rookie employees may be working more overtime, leading to burnout and increased turnover.

In [17]:
pd.crosstab(df["Age_Band"],df["overtime"])

overtime,No,Yes
Age_Band,,
Rookie(below 30),280,106
Experienced(30-50),675,266
Senior(50-60),99,44


In [18]:
pd.crosstab(
    df["Age_Band"],
    df["overtime"],
    normalize="index"
) * 100

overtime,No,Yes
Age_Band,,
Rookie(below 30),72.538860,27.461140
Experienced(30-50),71.732200,28.267800
Senior(50-60),69.230769,30.769231


### Finding

Overtime participation is similar across all age groups.

Rookie employees: ~27%
Experienced employees: ~28%
Senior employees: ~31%

The difference is relatively small and does not appear to explain the higher attrition observed among rookie employees.

Job Satisfaction

In [19]:
df.groupby("Age_Band")["JobSatisfaction"].mean()

/tmp/ipykernel_4812/734409333.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Age_Band")["JobSatisfaction"].mean()


,JobSatisfaction
Age_Band,
Rookie(below 30),2.704663
Experienced(30-50),2.733262
Senior(50-60),2.762238


### Finding

Average job satisfaction scores are very similar across all age groups.

Rookie employees: 2.70
Experienced employees: 2.73
Senior employees: 2.76

Job satisfaction does not appear to vary enough to explain the higher attrition among rookie employees.

## Limitations

This analysis focused on age, salary, overtime, and job satisfaction.

Other factors such as department, business travel, work-life balance, years at company, and career growth opportunities were not investigated and may also influence employee attrition.

The findings identify associations within the data and should not be interpreted as proof of causation.

## Final Conclusion

The overall attrition rate was found to be 16%.

Employees aged 18-30 (Rookies) showed the highest attrition rate at 25%, compared to 12% for experienced and senior employees.

Overtime and job satisfaction showed little variation across groups and did not appear to explain the higher attrition among rookies.

Salary showed the strongest association with attrition among the factors investigated.

Rookie employees who left the company earned significantly less than those who stayed, suggesting compensation may be an important factor associated with attrition.

In [20]:
df.to_csv("employee_attrition_analysis.csv", index=False)